# 01 — Obtenção e carregamento dos dados

**Objetivo:** baixar os microdados de acidentes da PRF (2017-2025), ler os arquivos com a codificação correta e gerar a base consolidada em Parquet.

**Fonte oficial:** [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf), arquivo *Agrupados por ocorrência* (`datatran<ano>.csv`). Uma linha = um acidente.

Código: `src/data_processing.py`.

In [1]:
# Configuração: roda a partir da raiz do repositório
import os, sys, warnings
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
from src import config

## 1. Download

O download oficial é feito pelo `gdown` a partir dos IDs do Google Drive listados na página da PRF (`config.PRF_DRIVE_IDS`).
Deixe `BAIXAR = True` na primeira execução local. Os resultados deste repositório usaram cópias públicas dos mesmos arquivos (ver `data/README.md` e `data/raw/manifest.json`).

In [2]:
BAIXAR = False  # True para baixar da fonte oficial (requer internet)
from src import data_processing as dp
if BAIXAR:
    for ano in config.ANOS:
        dp.baixar_ano(ano)
    dp.gravar_manifest(config.ANOS, origem="Google Drive oficial da PRF")
manifest = json.loads((config.RAW_DIR / "manifest.json").read_text())
pd.DataFrame(manifest["arquivos"])[["ano", "arquivo", "bytes", "espelho_github"]]

,ano,arquivo,bytes,espelho_github
0,2017,datatran2017.csv,24026333,FelipeGBarreto/analise_cluster_acidentes_rodov...
1,2018,datatran2018.csv,18803451,FelipeGBarreto/analise_cluster_acidentes_rodov...
2,2019,datatran2019.csv,18129710,FelipeGBarreto/analise_cluster_acidentes_rodov...
3,2020,datatran2020.csv,17191029,FelipeGBarreto/analise_cluster_acidentes_rodov...
4,2021,datatran2021.csv,18029712,FelipeGBarreto/analise_cluster_acidentes_rodov...
5,2022,datatran2022.csv,18723306,LeticiaHms/mack-car-crash-prediction
6,2023,datatran2023.csv,19770946,LeticiaHms/mack-car-crash-prediction
7,2024,datatran2024.csv,21644260,Alli-will/content-datatran2024
8,2025,datatran2025.csv,21070906,Nickolas3211/datatran2025


## 2. Leitura de um ano: estrutura original

In [3]:
bruto_2025 = dp.ler_ano(2025)
print(bruto_2025.shape)
bruto_2025.head(3).T

(72529, 31)


,0,1,2
id,652493,652519,652522
data_inversa,2025-01-01,2025-01-01,2025-01-01
dia_semana,quarta-feira,quarta-feira,quarta-feira
horario,06:20:00,07:50:00,08:45:00
uf,SP,CE,PR
br,116,116,369
km,225,"546,2","88,2"
municipio,GUARULHOS,PENAFORTE,CORNELIO PROCOPIO
causa_acidente,Reação tardia ou ineficiente do condutor,Pista esburacada,Reação tardia ou ineficiente do condutor
tipo_acidente,Tombamento,Colisão frontal,Colisão traseira


**Pontos de atenção na leitura** (tratados em `ler_ano` e `limpar`):

- codificação **Latin-1** e separador `;`;
- separador decimal de latitude/longitude **muda entre anos** (vírgula em uns, ponto em outros): por isso tudo é lido como texto e convertido por `_num`;
- `tracado_via` é multivalorado (`"Reta;Declive"`).

In [4]:
bruto = dp.ler_anos(config.ANOS)
resumo = (bruto.groupby("ano_arquivo")["classificacao_acidente"].value_counts(normalize=True)
          .unstack().mul(100).round(1))
resumo.insert(0, "acidentes", bruto.groupby("ano_arquivo").size())
resumo

classificacao_acidente,acidentes,Com Vítimas Fatais,Com Vítimas Feridas,NA,Sem Vítimas
ano_arquivo,,,,,
2017,89563,5.8,60.0,NaN,34.3
2018,69295,6.5,71.5,NaN,22.0
2019,67446,6.8,75.9,NaN,17.3
2020,63548,7.1,74.6,NaN,18.2
2021,64515,7.2,74.7,NaN,18.1
2022,64606,7.2,75.0,0.0,17.8
2023,67766,7.2,76.7,0.0,16.2
2024,73156,7.1,76.8,0.0,16.1
2025,72529,7.2,77.5,0.0,15.4


## 3. Limpeza, harmonização e base consolidada

In [5]:
base, log = dp.construir_base(config.ANOS)
print(json.dumps(log, indent=2, ensure_ascii=False))
print(base.shape)
base.head(3).T

{
  "linhas_lidas": 632424,
  "sem_alvo_valido": 4,
  "duplicatas_id": 0,
  "data_ou_hora_invalida": 0,
  "coordenadas_fora_do_brasil": 54,
  "pessoas_diferente_da_soma": 15902,
  "linhas_finais": 632420
}
(632420, 48)


,0,1,2
id,1091,11,12
data,2017-01-01 00:00:00,2017-01-01 00:00:00,2017-01-01 00:00:00
ano,2017,2017,2017
hora,0.0,0.0,0.0
municipio,RIO DE JANEIRO,SANTO ANTONIO DA PLATINA,ANAPOLIS
br,40,153,153
causa_acidente,Falta de Atenção do Pedestre,Animais na Pista,Avarias e/ou desgaste excessivo no pneu
condicao_meteo,Nublado,Garoa/Chuvisco,Céu Claro
dia_semana,domingo,domingo,domingo
faixa_horaria,madrugada,madrugada,madrugada


A base consolidada (`data/processed/acidentes_2017_2025.parquet`) é a entrada de todos os notebooks seguintes.

**Próximo passo:** `02_analise_exploratoria.ipynb`.